# 06 — Models

Architecture definitions only — no training loop, no normalization
(both are `07`'s job). Builds every scenario (A-E; F deferred; G separate)
and runs one real forward pass per scenario against an actual graph pair
from `05`'s index, to catch shape/dimension bugs before a full `07` run.

Uses `src/models.py`, `src/baseline_features.py`.

In [ ]:
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric pyyaml pandas

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

PROCESSED_DIR = Path(paths_cfg["processed_dir"])
SVG_DIR = PROCESSED_DIR / "svg_graphs"
TVG_DIR = PROCESSED_DIR / "tvg_graphs"

# Vocab sizes — fixed by your actual data, not tunable
SIGNAGE_VOCAB = 5
LIGHT_POLE_VOCAB = 4
ROAD_MARKING_VOCAB = 2
BUILDING_TYPE_VOCAB = 58
HIGHWAY_VOCAB = 13

# Embedding dims: min(8, max(2, ceil(vocab/4)))
EMBED_DIMS = {
    "signage": 2, "light_pole": 2, "road_marking": 2,
    "building_type": 8, "highway": 4,
}
print("Embedding dims:", EMBED_DIMS)

In [ ]:
import models
import torch

svg_kwargs = dict(
    hidden_dim=64, heads=4, num_layers=2, dropout=0.35,
    signage_vocab=SIGNAGE_VOCAB, light_pole_vocab=LIGHT_POLE_VOCAB,
    road_marking_vocab=ROAD_MARKING_VOCAB, cat_embed_dim=EMBED_DIMS["signage"],
)
tvg_kwargs = dict(
    hidden_dim=64, heads=4, num_layers=2, dropout=0.35,
    building_type_vocab=BUILDING_TYPE_VOCAB, highway_vocab=HIGHWAY_VOCAB,
    building_type_embed_dim=EMBED_DIMS["building_type"], highway_embed_dim=EMBED_DIMS["highway"],
)

In [ ]:
# ── Build every scenario (A-E; F deferred; G is separate/no torch model) ──
SCENARIOS = ["A", "B", "C", "D", "E"]
HEAD_DEPTHS = ["linear", "mlp2"]  # both compared, per the earlier decision

built_models = {}
for scenario in SCENARIOS:
    for depth in HEAD_DEPTHS:
        key = f"{scenario}_{depth}"
        built_models[key] = models.build_model(
            scenario, fusion_dim=64, head_depth=depth,
            use_ablation=False, svg_kwargs=svg_kwargs, tvg_kwargs=tvg_kwargs,
        )
        n_params = sum(p.numel() for p in built_models[key].parameters())
        print(f"{key:10s} — {n_params:,} parameters")

In [ ]:
# ── QC: one real forward pass per scenario against an actual graph pair ──
# Catches shape/dimension bugs now, not 20 minutes into a 07 training run.
index_df = None
try:
    import pandas as pd
    index_df = pd.read_parquet(PROCESSED_DIR / "dataset_index.parquet")
except FileNotFoundError:
    print("dataset_index.parquet not found yet — run 05 first for this QC cell to work.")

if index_df is not None:
    sample_pid = index_df["point_id"].iloc[0]
    svg_sample = torch.load(SVG_DIR / f"{sample_pid}.pt", weights_only=False)
    tvg_sample = torch.load(TVG_DIR / f"{sample_pid}.pt", weights_only=False)
    print(f"Testing against: {sample_pid}")

    for scenario in SCENARIOS:
        for depth in HEAD_DEPTHS:
            key = f"{scenario}_{depth}"
            model = built_models[key]
            model.eval()
            with torch.no_grad():
                try:
                    if scenario == "A":
                        out = model(svg_sample)
                    elif scenario == "B":
                        out = model(tvg_sample)
                    else:
                        out = model(svg_sample, tvg_sample)
                    print(f"  ✅ {key}: output shape {tuple(out.shape)}, value {out.item():.4f}")
                except Exception as e:
                    print(f"  ❌ {key}: {type(e).__name__}: {e}")

In [ ]:
# ── Scenario G: build the tabular feature table (separate path, no torch) ──
import baseline_features

if index_df is not None:
    sample_ids = index_df["point_id"].tolist()[:5]  # small sample for this QC check
    feat_table = baseline_features.build_feature_table(sample_ids, SVG_DIR, TVG_DIR, torch)
    display(feat_table)

In [ ]:
print("Scenario F (unified merged graph) intentionally deferred — build")
print("and validate A-E first, per the established plan.")
print()
print("Still open for 07: batch size, epoch cap, patience, per-fold")
print("normalization fitting, and which head depth (linear vs mlp2) wins.")
print()
print("Next: 07_train_eval.ipynb")